# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [38]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage, BaseMessage
from lib.tooling import tool

In [40]:
from dotenv import load_dotenv
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
import chromadb
from typing import List, Dict, Union
from lib.llm import LLM
from lib.rag import RAG
from lib.vector_db import VectorStore
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
chroma_llm = LLM()
chroma_rag = RAG(llm=chroma_llm, vector_store=collection)
chroma_vdb = VectorStore(collection)
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str): 
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """

    game_run = chroma_vdb.query(query_texts=query)

    return  game_run["metadatas"][0]

In [ ]:
# Test the retrieve_game tool
retrieve_game(query="Gran Turismo")

[{'Name': 'Gran Turismo 5',
  'YearOfRelease': 2010,
  'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.',
  'Platform': 'PlayStation 3',
  'Publisher': 'Sony Computer Entertainment',
  'Genre': 'Racing'},
 {'Publisher': 'Sony Computer Entertainment',
  'YearOfRelease': 1997,
  'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
  'Name': 'Gran Turismo',
  'Platform': 'PlayStation 1',
  'Genre': 'Racing'},
 {'YearOfRelease': 2004,
  'Description': "An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.",
  'Publisher': 'Rockstar Games',
  'Genre': 'Action-adventure',
  'Name': 'Grand Theft Auto: San Andreas',
  'Platform': 'PlayStation 2'}]

#### Evaluate Retrieval Tool

In [29]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result


from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser

class EvaluationReport(BaseModel):
    """Structured report from the retrieval evaluation"""
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Description about the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge_llm = LLM(model="gpt-4o-mini")

    judge_prompt = (
        f"You are an expert retrieval evaluator.\n\n"
        f"User question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}\n\n"
        f"Your task is to evaluate if the documents are enough to respond the query. "
        f"Give a detailed explanation, so it's possible to take an action to accept it or not."
    )

    response = judge_llm.invoke(
        input=judge_prompt,
        response_format=EvaluationReport,
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return report.model_dump_json()


In [ ]:
# Test the evaluate_retrieval
docs = retrieve_game(query="What platforms can I play gran turismo on?")
evaluate_retrieval("Gran Turismo", docs)

'{"useful":true,"description":"The retrieved documents provide relevant information about the \'Gran Turismo\' franchise, specifically focusing on two key titles: \'Gran Turismo 5\' and the original \'Gran Turismo\' released in 1997. Both documents include essential details such as the platform, genre, publisher, year of release, and a brief description of each game. This information is directly related to the user query about \'Gran Turismo\', making the documents useful for answering the question. \\n\\nHowever, the third document about \'Mario Kart 8 Deluxe\' is not relevant to the query as it pertains to a different racing game franchise. Despite this, the presence of two relevant documents is sufficient to provide a comprehensive overview of the \'Gran Turismo\' series, allowing the user to gain insights into its history and evolution. Therefore, the overall evaluation concludes that the documents are useful for responding to the query."}'

#### Game Web Search Tool

In [31]:
from tavily import TavilyClient
from datetime import datetime

# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question):
   """
   Tool Docstring:
      Semantic search: Finds most results in the vector DB
      args:
      - question: a question about game industry. 
   """
   client = TavilyClient(api_key=TAVILY_API_KEY)

   # Perform the search
   search_result = client.search(
      query=question,
      include_answer=True,
      include_raw_content=False,
      include_images=False
   )
   
   # Format the results
   formatted_results = {
      "answer": search_result.get("answer", ""),
      "top_3_results": search_result.get("results", [])[:3],
      "search_metadata": {
         "timestamp": datetime.now().isoformat(),
         "query": question
      }
   }
   
   return formatted_results

In [ ]:
# Test game_web_search tool
game_web_search("What platforms can I play gran turismo on?")

{'answer': 'Gran Turismo is available on PlayStation consoles. Gran Turismo 7 is also purchasable via the PlayStation Store. Emulators can run older versions on PC.',
 'top_3_results': [{'url': 'https://en.wikipedia.org/wiki/Gran_Turismo_(series)',
   'title': 'Gran Turismo (series) - Wikipedia',
   'content': '# *Gran Turismo* (series). * Edit&action=edit "Edit this page [e]"). Series of racing video games. ***Gran Turismo*** (***GT***) is a series of sim racing video games developed by Polyphony Digital. Released for PlayStation systems, *Gran Turismo* games are intended to emulate the appearance and performance of a large selection of vehicles, most of which are licensed reproductions of real-world automobiles. Notable competitions held on *Gran Turismo* include the Gran Turismo World Series and the former GT Academy. The first title "Gran Turismo (1997 video game)") in the series was the highest selling game for the original PlayStation, while four subsequent installments have been

### Agent

In [33]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [34]:
tools = [retrieve_game, evaluate_retrieval, game_web_search]

In [46]:
game_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You're a gaming analytics research agent that produces clear, structured, and well-cited answers. "
        "When answering questions or queries, make sure you first check using the retrieve game tool. "
        "After checking with the tool, you're to use the evaluation tool to evaluate the output to see if it's relevant to the question. "
        "If the response from the tool is relevant, you can use it to produce an answer, otherwise, result to using the web search tool. "
        "If the web search tool is used, use it to produce the final response to the user."
        ),
    tools=tools
)

In [ ]:
import json as _json
import textwrap

def print_messages(messages: List[BaseMessage]):
    ICONS = {'system': '⚙️', 'user': '👤', 'assistant': '🤖', 'tool': '🔧'}
    SEP = '─' * 60

    for i, m in enumerate(messages):
        icon = ICONS.get(m.role, '❓')
        header = m.role.upper()

        # Add tool name to header for tool messages
        if isinstance(m, ToolMessage):
            header += f'  [{m.name}]'

        print(f'\n{SEP}')
        print(f'{icon}  {header}')
        print(SEP)

        # Print content
        if m.content:
            # Try to pretty-print JSON content
            try:
                parsed = _json.loads(m.content)
                pretty = _json.dumps(parsed, indent=2, ensure_ascii=False)
                if len(pretty) > 800:
                    pretty = pretty[:800] + '\n... [truncated]'
                print(pretty)
            except (ValueError, TypeError):
                content = m.content
                if len(content) > 1000:
                    content = content[:1000] + '... [truncated]'
                print(content)

        # Print tool calls for assistant messages
        tool_calls = getattr(m, 'tool_calls', None)
        if tool_calls:
            for tc in tool_calls:
                try:
                    args = _json.loads(tc.function.arguments)
                    args_str = _json.dumps(args, indent=2, ensure_ascii=False)
                except (ValueError, TypeError):
                    args_str = tc.function.arguments
                print(f'  📞 Call: {tc.function.name}')
                print(textwrap.indent(args_str, '     '))

    print(f'\n{"═" * 60}')
    print(f'  Total messages: {len(messages)}')
    print(f'{"═" * 60}')


In [36]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

In [ ]:
# Create new session
run1 = game_agent.invoke(query="When Pokémon Gold and Silver was released?", session_id="game_query")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [51]:
print_messages(run1.get_final_state()["messages"])


────────────────────────────────────────────────────────────
⚙️  SYSTEM
────────────────────────────────────────────────────────────
You're a gaming analytics research agent that produces clear, structured, and well-cited answers. When answering questions or queries, make sure you first check using the retrieve game tool. After checking with the tool, you're to use the evaluation tool to evaluate the output to see if it's relevant to the question. If the response from the tool is relevant, you can use it to produce an answer, otherwise, result to using the web search tool. If the web search tool is used, use it to produce the final response to the user.

────────────────────────────────────────────────────────────
👤  USER
────────────────────────────────────────────────────────────
When Pokémon Gold and Silver was released?

────────────────────────────────────────────────────────────
🤖  ASSISTANT
────────────────────────────────────────────────────────────
  📞 Call: retrieve_game
   

In [ ]:
# Still on the same game_query session
run1 = game_agent.invoke(query="Which one was the first 3D platformer Mario game?", session_id="game_query")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [ ]:
# Conversation should show previous context from the session
print_messages(run1.get_final_state()["messages"])


────────────────────────────────────────────────────────────
⚙️  SYSTEM
────────────────────────────────────────────────────────────
You're a gaming analytics research agent that produces clear, structured, and well-cited answers. When answering questions or queries, make sure you first check using the retrieve game tool. After checking with the tool, you're to use the evaluation tool to evaluate the output to see if it's relevant to the question. If the response from the tool is relevant, you can use it to produce an answer, otherwise, result to using the web search tool. If the web search tool is used, use it to produce the final response to the user.

────────────────────────────────────────────────────────────
👤  USER
────────────────────────────────────────────────────────────
When Pokémon Gold and Silver was released?

────────────────────────────────────────────────────────────
🤖  ASSISTANT
────────────────────────────────────────────────────────────
  📞 Call: retrieve_game
   

In [ ]:
# Run with a new session
run1 = game_agent.invoke(query="Was Mortal Kombat X realeased for Playstation 5?", , session_id="game_query_2")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [55]:
print_messages(run1.get_final_state()["messages"])


────────────────────────────────────────────────────────────
⚙️  SYSTEM
────────────────────────────────────────────────────────────
You're a gaming analytics research agent that produces clear, structured, and well-cited answers. When answering questions or queries, make sure you first check using the retrieve game tool. After checking with the tool, you're to use the evaluation tool to evaluate the output to see if it's relevant to the question. If the response from the tool is relevant, you can use it to produce an answer, otherwise, result to using the web search tool. If the web search tool is used, use it to produce the final response to the user.

────────────────────────────────────────────────────────────
👤  USER
────────────────────────────────────────────────────────────
When Pokémon Gold and Silver was released?

────────────────────────────────────────────────────────────
🤖  ASSISTANT
────────────────────────────────────────────────────────────
  📞 Call: retrieve_game
   

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes